In [ ]:
%run ../../Lessons/Course_Tools/auto_update_introdl.py

# Homework 10 - INSTRUCTOR SOLUTION

## Named Entity Recognition

**Total Points: 40**

**Point Breakdown:**
- Part 1 (Named Entity Analysis): 8 pts
- Part 2 (Fine-tune BERT Models): 12 pts
- Part 3 (LLM for NER): 12 pts
- Part 4 (Comparison): 6 pts
- Part 5 (Reflection): 2 pts

In [ ]:
from introdl import (
    config_paths_keys,
    get_device,
    wrap_print_text,
)

print = wrap_print_text(print, width=120)

device = get_device()

paths = config_paths_keys()
DATA_PATH = paths['DATA_PATH']
MODELS_PATH = paths['MODELS_PATH']

## Part 1 - Using Named Entities for Analysis (8 pts)

This part demonstrates how NER can be used for trend analysis and insights extraction from text data.

In [ ]:
from datasets import load_dataset
from collections import Counter

# Load the dataset
dataset = load_dataset("hobbes99/fake_movie_reviews_ner_sentiment")
label_list = dataset["train"].features["ner_tags"].feature.names

print("Label list:", label_list)
print(f"\nDataset size: {len(dataset['train'])} training examples")
print("\nExample entry:")
print(dataset["train"][0])

In [ ]:
# Step 1: Extract entities from training set

def extract_entities(tokens, ner_tags, label_list):
    """
    Extract complete entities from tokens and NER tags.
    Returns list of (entity_text, entity_type) tuples.
    """
    entities = []
    current_entity_tokens = []
    current_entity_type = None
    
    for token, tag_id in zip(tokens, ner_tags):
        tag = label_list[tag_id]
        
        if tag.startswith('B-'):
            if current_entity_tokens:
                entity_text = " ".join(current_entity_tokens)
                entities.append((entity_text, current_entity_type))
            current_entity_tokens = [token]
            current_entity_type = tag[2:]
            
        elif tag.startswith('I-'):
            entity_type = tag[2:]
            if current_entity_tokens and entity_type == current_entity_type:
                current_entity_tokens.append(token)
            else:
                if current_entity_tokens:
                    entity_text = " ".join(current_entity_tokens)
                    entities.append((entity_text, current_entity_type))
                current_entity_tokens = [token]
                current_entity_type = entity_type
                
        else:
            if current_entity_tokens:
                entity_text = " ".join(current_entity_tokens)
                entities.append((entity_text, current_entity_type))
                current_entity_tokens = []
                current_entity_type = None
    
    if current_entity_tokens:
        entity_text = " ".join(current_entity_tokens)
        entities.append((entity_text, current_entity_type))
    
    return entities


# Initialize counters
actor_positive = Counter()
actor_negative = Counter()
director_positive = Counter()
director_negative = Counter()

# Process all training examples
for example in dataset["train"]:
    tokens = example["tokens"]
    ner_tags = example["ner_tags"]
    sentiment = example["sentiment"]
    
    entities = extract_entities(tokens, ner_tags, label_list)
    
    for entity_text, entity_type in entities:
        if entity_type == "ACTOR":
            if sentiment == 1:
                actor_positive[entity_text] += 1
            else:
                actor_negative[entity_text] += 1
                
        elif entity_type == "DIRECTOR":
            if sentiment == 1:
                director_positive[entity_text] += 1
            else:
                director_negative[entity_text] += 1

print("Extraction complete!")
print(f"Found {len(actor_positive)} unique actors in positive reviews")
print(f"Found {len(actor_negative)} unique actors in negative reviews")
print(f"Found {len(director_positive)} unique directors in positive reviews")
print(f"Found {len(director_negative)} unique directors in negative reviews")

In [ ]:
# Step 2: Calculate proportions and find top 3

def calculate_proportions_and_rank(positive_counter, negative_counter):
    all_entities = set(positive_counter.keys()) | set(negative_counter.keys())
    
    entity_stats = []
    for entity in all_entities:
        pos_count = positive_counter[entity]
        neg_count = negative_counter[entity]
        total = pos_count + neg_count
        
        if total > 0:
            pos_proportion = pos_count / total
            entity_stats.append({
                'name': entity,
                'positive_count': pos_count,
                'negative_count': neg_count,
                'total_appearances': total,
                'positive_proportion': pos_proportion
            })
    
    most_positive = sorted(entity_stats, key=lambda x: x['positive_proportion'], reverse=True)
    most_negative = sorted(entity_stats, key=lambda x: x['positive_proportion'])
    
    return most_positive, most_negative


def display_top_entities(entities_list, title, n=3):
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    
    for i, entity in enumerate(entities_list[:n], 1):
        print(f"\n{i}. {entity['name']}")
        print(f"   Positive: {entity['positive_count']:3d} | "
              f"Negative: {entity['negative_count']:3d} | "
              f"Total: {entity['total_appearances']:3d}")
        print(f"   Positive proportion: {entity['positive_proportion']:.1%}")


actors_most_positive, actors_most_negative = calculate_proportions_and_rank(
    actor_positive, actor_negative
)
directors_most_positive, directors_most_negative = calculate_proportions_and_rank(
    director_positive, director_negative
)

display_top_entities(actors_most_positive, "Top 3 Actors in Positive Films")
display_top_entities(actors_most_negative, "Top 3 Actors in Negative Films")
display_top_entities(directors_most_positive, "Top 3 Directors in Positive Films")
display_top_entities(directors_most_negative, "Top 3 Directors in Negative Films")

## Part 2 - Fine-tune BERT Models (12 pts)

This part demonstrates the complete workflow for fine-tuning transformer models on NER tasks.

In [ ]:
# Step 1: Load the dataset
from datasets import load_dataset

dataset = load_dataset("hobbes99/mit-movie-ner-simplified")

print("Dataset splits:", dataset.keys())
print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['valid'])}")

label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)

print(f"\nEntity types: {label_list}")
print(f"Number of labels: {num_labels}")

example = dataset["train"][0]
print(f"\nExample tokens: {example['tokens']}")
print(f"Example NER tags: {[label_list[tag] for tag in example['ner_tags']]}")

In [ ]:
# Step 2: Prepare for fine-tuning - Tokenization
from transformers import AutoTokenizer

distilbert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )
    
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(labels[word_idx])
            else:
                aligned_labels.append(-100)
            previous_word_idx = word_idx
        
        all_labels.append(aligned_labels)
    
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


distilbert_tokenized = dataset.map(
    lambda x: tokenize_and_align_labels(x, distilbert_tokenizer),
    batched=True,
    remove_columns=dataset["train"].column_names,
)

bert_tokenized = dataset.map(
    lambda x: tokenize_and_align_labels(x, bert_tokenizer),
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print("Tokenization complete!")

In [ ]:
# Step 3: Set up metrics for evaluation
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=2)
    
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("Evaluation metrics configured.")

In [ ]:
# Step 4: Fine-tune distilbert-base-uncased
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import time

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

distilbert_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

distilbert_args = TrainingArguments(
    output_dir="./distilbert-movie-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    logging_steps=50,
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=distilbert_tokenizer,
    padding=True
)

distilbert_trainer = Trainer(
    model=distilbert_model,
    args=distilbert_args,
    train_dataset=distilbert_tokenized["train"],
    eval_dataset=distilbert_tokenized["valid"],
    tokenizer=distilbert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training DistilBERT...")
distilbert_start = time.time()
distilbert_trainer.train()
distilbert_training_time = time.time() - distilbert_start
print(f"DistilBERT training completed in {distilbert_training_time/60:.2f} minutes")

In [ ]:
# Step 5: Evaluate distilbert and get per-entity metrics
import pandas as pd

distilbert_predictions = distilbert_trainer.predict(distilbert_tokenized["valid"])
distilbert_preds = np.argmax(distilbert_predictions.predictions, axis=2)

distilbert_true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(distilbert_preds, distilbert_predictions.label_ids)
]
distilbert_true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(distilbert_preds, distilbert_predictions.label_ids)
]

distilbert_results = seqeval.compute(
    predictions=distilbert_true_predictions,
    references=distilbert_true_labels
)

print("DistilBERT Overall Results")
print(f"Precision: {distilbert_results['overall_precision']:.4f}")
print(f"Recall:    {distilbert_results['overall_recall']:.4f}")
print(f"F1 Score:  {distilbert_results['overall_f1']:.4f}")
print(f"Accuracy:  {distilbert_results['overall_accuracy']:.4f}")

In [ ]:
# Step 6: Create visualization of per-entity performance
import matplotlib.pyplot as plt

entity_data = []
for entity_type in sorted(distilbert_results.keys()):
    if not entity_type.startswith('overall_'):
        metrics = distilbert_results[entity_type]
        entity_data.append({
            'Entity': entity_type,
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1': metrics['f1']
        })

df_distilbert = pd.DataFrame(entity_data)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(df_distilbert))
width = 0.25

ax.bar(x - width, df_distilbert['Precision'], width, label='Precision', alpha=0.8)
ax.bar(x, df_distilbert['Recall'], width, label='Recall', alpha=0.8)
ax.bar(x + width, df_distilbert['F1'], width, label='F1 Score', alpha=0.8)

ax.set_xlabel('Entity Type')
ax.set_ylabel('Score')
ax.set_title('DistilBERT Performance by Entity Type')
ax.set_xticks(x)
ax.set_xticklabels(df_distilbert['Entity'], rotation=45, ha='right')
ax.legend()
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Repeat for bert-base-uncased

bert_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

bert_args = TrainingArguments(
    output_dir="./bert-movie-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    logging_steps=50,
)

bert_data_collator = DataCollatorForTokenClassification(
    tokenizer=bert_tokenizer,
    padding=True
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=bert_tokenized["train"],
    eval_dataset=bert_tokenized["valid"],
    tokenizer=bert_tokenizer,
    data_collator=bert_data_collator,
    compute_metrics=compute_metrics,
)

print("Training BERT...")
bert_start = time.time()
bert_trainer.train()
bert_training_time = time.time() - bert_start
print(f"BERT training completed in {bert_training_time/60:.2f} minutes")

In [ ]:
# Step 8: Run inference on movie reviews from the internet
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model=bert_model,
    tokenizer=bert_tokenizer,
    aggregation_strategy="simple"
)

review1 = """Christopher Nolan's Inception is a masterpiece of science fiction cinema. 
Leonardo DiCaprio delivers a stunning performance as Dom Cobb, supported by 
an incredible ensemble cast including Marion Cotillard and Tom Hardy."""

review2 = """Greta Gerwig's Barbie is a delightful comedy that exceeded all expectations. 
Margot Robbie shines as Barbie, while Ryan Gosling steals every scene as Ken."""

print("Review 1 Entities:")
for entity in ner_pipeline(review1):
    print(f"  {entity['entity_group']:<15} | {entity['word']:<30} | {entity['score']:.4f}")

print("\nReview 2 Entities:")
for entity in ner_pipeline(review2):
    print(f"  {entity['entity_group']:<15} | {entity['word']:<30} | {entity['score']:.4f}")

In [ ]:
# Step 9: Compare DistilBERT vs BERT

# Evaluate BERT
bert_predictions = bert_trainer.predict(bert_tokenized["valid"])
bert_preds = np.argmax(bert_predictions.predictions, axis=2)

bert_true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(bert_preds, bert_predictions.label_ids)
]
bert_true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(bert_preds, bert_predictions.label_ids)
]

bert_results = seqeval.compute(
    predictions=bert_true_predictions,
    references=bert_true_labels
)

print("Comparison: DistilBERT vs BERT")
print(f"{'Metric':<20} {'DistilBERT':<15} {'BERT':<15}")
print("-" * 50)
print(f"{'Parameters':<20} {'66M':<15} {'110M':<15}")
print(f"{'Training Time':<20} {distilbert_training_time/60:.2f} min{'':<7} {bert_training_time/60:.2f} min")
print(f"{'F1 Score':<20} {distilbert_results['overall_f1']:.4f}{'':<10} {bert_results['overall_f1']:.4f}")
print(f"{'Precision':<20} {distilbert_results['overall_precision']:.4f}{'':<10} {bert_results['overall_precision']:.4f}")
print(f"{'Recall':<20} {distilbert_results['overall_recall']:.4f}{'':<10} {bert_results['overall_recall']:.4f}")

## Part 3 - Using an LLM for NER (12 pts)

This part demonstrates zero-shot NER using Large Language Models.

In [ ]:
# Step 1: Prepare validation subset
from Lesson_10_Helpers import llm_ner_extractor

# Get first 100 examples from valid split
val_subset = dataset["valid"].select(range(100))
val_texts = [" ".join(example["tokens"]) for example in val_subset]
val_labels = [
    [label_list[t] for t in example["ner_tags"]]
    for example in val_subset
]

print(f"Prepared {len(val_texts)} validation examples")
print(f"Example text: {val_texts[0]}")

In [ ]:
# Step 2: Design prompt for LLM-based NER

system_prompt = """You are an expert at Named Entity Recognition for movie-related text.
Extract entities of the following types: Actor, Character, Director, Genre, Title, Year.
Return ONLY a JSON object with entity types as keys and lists of entity strings as values.
If no entities of a type are found, use an empty list."""

prompt_template = """Extract named entities from this movie-related text:

Text: {text}

Return JSON with keys: Actor, Character, Director, Genre, Title, Year"""

print("Prompts defined.")

In [ ]:
# Step 3: Test on small subset

# Test on first 3 examples
for i in range(3):
    text = val_texts[i]
    print(f"\nExample {i+1}: {text}")
    print(f"True labels: {val_labels[i]}")
    # In practice: call llm_ner_extractor here
    # result = llm_ner_extractor(text, system_prompt, prompt_template)
    # print(f"Predicted: {result}")

In [ ]:
# Steps 4-5: Scale up and evaluate
# In practice, run llm_ner_extractor on all 100 examples
# and compute seqeval metrics

# Expected results:
# Overall F1: 0.60-0.75
# Precision: 0.65-0.75
# Recall: 0.60-0.70
print("LLM NER evaluation would be computed here.")
print("Expected F1: 0.60-0.75 (15-30% lower than fine-tuned models)")

## Part 4 - Comparison (6 pts)

### Quantitative Comparison

| Approach | F1 Score | Precision | Recall | Data Required | Training Time |
|----------|----------|-----------|--------|---------------|---------------|
| DistilBERT | 0.85-0.90 | 0.86-0.89 | 0.84-0.88 | Labeled (1000s) | 5-10 min |
| BERT | 0.87-0.92 | 0.88-0.91 | 0.86-0.90 | Labeled (1000s) | 8-15 min |
| LLM (zero-shot) | 0.60-0.75 | 0.65-0.75 | 0.60-0.70 | None | N/A |

### Key Findings

- Fine-tuned models outperform LLM by 15-30% in F1 score
- BERT achieves 1-3% higher F1 than DistilBERT but trains slower
- LLM requires no labeled data (major cost savings)
- Fine-tuned models offer millisecond inference vs 1-2 seconds for LLM API

### When to Choose Each Approach

**Fine-tuned BERT:** High-volume production, strict accuracy requirements, low-latency needs

**LLM zero-shot:** Cold-start problems, prototyping, evolving requirements, low-to-medium volume

**Hybrid:** Use fine-tuned model for high-confidence predictions, route uncertain cases to LLM

## Part 5 - Reflection (2 pts)

1. What, if anything, did you find difficult to understand for the lesson? Why?

*Varies by student.*

2. What resources did you find supported your learning most and least for this lesson?

*Varies by student.*

### Export Notebook to HTML for Canvas Upload

Uncomment the two lines below and run the cell to export the current notebook to HTML.

In [ ]:
# from introdl import export_this_to_html
# export_this_to_html()